# Re-train the gate detector (yolo11s, 1-class `gate`)
**Before running:** Runtime → Change runtime type → **GPU (T4)**, then Runtime → **Run all**.

Trains from the recovered labeled dataset on branch `gate-detector-retrain-probe` (public repo). Recipe: imgsz 960, freeze 10, cos-lr, patience=30 early-stop, epochs capped 100. Downloads `best.pt` at the end.

In [ ]:
!pip -q install "ultralytics>=8.0.0"
import ultralytics; ultralytics.checks()   # confirms GPU is active

In [ ]:
# Public repo — clone the branch that carries the dataset
!git clone --branch gate-detector-retrain-probe --single-branch https://github.com/smiky2011/Skiing.git /content/Skiing
DATA_ROOT = "/content/Skiing/gate_detection/data/datasets/final_combined_1class_20260215"
import os; print('exists:', os.path.isdir(DATA_ROOT))

In [ ]:
# Inject an absolute path so Ultralytics resolves the splits
import yaml, os
yml = os.path.join(DATA_ROOT, "data.yaml")
d = yaml.safe_load(open(yml))
d["path"] = DATA_ROOT
d["train"], d["val"], d["test"] = "train/images", "valid/images", "test/images"
yaml.safe_dump(d, open(yml, "w"))
print(d)   # expect nc:1, names:['gate']

In [ ]:
# Train (yolo11s detection base; recovered recipe; epochs capped 100, early-stops via patience)
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.train(
    data=yml, epochs=100, imgsz=960, batch=16, freeze=10, cos_lr=True,
    patience=30, close_mosaic=25,
    flipud=0.0, fliplr=0.5, mosaic=0.5, mixup=0.0, copy_paste=0.0,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, scale=0.5,
    project="/content/runs_gate", name="gate_yolo11s", device=0, workers=8,
)
BEST = "/content/runs_gate/gate_yolo11s/weights/best.pt"
print("best:", BEST)
# OOM? set batch=8. Underfit at the cap? raise epochs.

In [ ]:
# Report BOTH val and test mAP
from ultralytics import YOLO
print("VAL :", YOLO(BEST).val(data=yml, split="val",  device=0).results_dict)
print("TEST:", YOLO(BEST).val(data=yml, split="test", device=0).results_dict)

In [ ]:
# Download the weights -> save locally as gate_yolo11s.pt for the probe
from google.colab import files
files.download(BEST)